**Importing the dataset**

In [12]:
from datasets import load_dataset

HumanEval_dataset = load_dataset("openai/openai_humaneval", split="test")
HumanEval_dataset

Dataset({
    features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
    num_rows: 164
})

**Filtering**

In [16]:
filter = HumanEval_dataset.to_pandas()[['task_id', 'prompt', 'canonical_solution']]
filter.head(5)

,task_id,prompt,canonical_solution
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,"for idx, elem in enumerate(numbers):\n ..."
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,result = []\n current_string = []\n ...
2,HumanEval/2,\n\ndef truncate_number(number: float) -> floa...,return number % 1.0\n
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...


**Embedding pipeline**

In [36]:
from sentence_transformers import SentenceTransformer
model_id = "sentence-transformers/distiluse-base-multilingual-cased-v2"
device = "cpu"
dim = 512
embeddingModel = SentenceTransformer(model_id, device="cpu")

In [ ]:
import chromadb
from chromadb.config import Settings

chromaClient = chromadb.PersistentClient(
    path="./chromadb-HumanEval-docs",
    settings=Settings(anonymized_telemetry=False)#===> saves to the permanent storage
)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


The current working directory is: c:\Users\Al Badr\Desktop\NU UNI Documents\Third Year Fall 2025\NLP Internship\Cellula_3week_[Yousef_Mahmoud_Ali]\Task


In [21]:
Prompts = filter['prompt'].tolist()

print("Embeddings...: ")
embeddings = model.encode(Prompts , show_progress_bar=True)
embeddings.shape

Embeddings...: 


Batches: 100%|██████████| 6/6 [00:04<00:00,  1.29it/s]


(164, 512)

In [22]:
collections = chromaClient.create_collection(
    name="HumanEval_dataset"
)
collections.add(
    embeddings=embeddings,
    documents=filter['prompt'].tolist(),
    ids= filter['task_id'].tolist()
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


**Code generation**

In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
codeGenModel = AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono")


Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

In [41]:
query = filter['prompt'].iloc[0]
queryTaskId = filter['task_id'].iloc[0]

query


#Retrieve
embiddingQuery = embeddingModel.encode([query])
rResults = collections.query(
    query_embeddings = embiddingQuery,
    n_results= 3 #top results
)
rPrompt = rResults['documents'][0]


#Augment
augmentedPrompt = "Test"
augmentedPrompt += "TEST2\n"
for i , prompt_text in enumerate(rPrompt):
    augmentedPrompt += f"Example {i+1}:\n```python\n{prompt_text}\n```\n\n"

augmentedPrompt += "Task\n"
augmentedPrompt += f"Complete the following Python code:\n```python\n{query}```"
print("\n\n--- Augmented Prompt Sent to Code Model ---")
print(augmentedPrompt)

#Generate
input = tokenizer(augmentedPrompt, return_tensors="pt")
output = codeGenModel.generate(**input, max_new_tokens = 100)

generatedCode = tokenizer.decode(output[0], skip_special_tokens=True)
print("\n\n--- Final Generated Code ---")
print(generatedCode)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




--- Augmented Prompt Sent to Code Model ---
TestTEST2
Example 1:
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

```

Example 2:
```python
from typing import List, Tuple


def find_closest_elements(numbers: List[float]) -> Tuple[float, float]:
    """ From a supplied list of numbers (of length at least two) select and return two that are the closest to each
    other and return them in order (smaller number, larger number).
    >>> find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.2])
    (2.0, 2.2)
    >>> find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.0])
    (2.0, 2.0)
    """

```

Example 3:
```python
from typing import List


def rescale_to_unit(numbers: List[float]) ->

**Integration**

In [ ]:
class CodeGenerationPipeline:
    def __init__(self, embedding_model, codegen_model, codegen_tokenizer, chroma_collection):

        print("Initializing pipeline with existing components...")
        self.embedding_model = embedding_model
        self.codegen_model = codegen_model
        self.codegen_tokenizer = codegen_tokenizer
        self.collection = chroma_collection  
        print("Pipeline initialized successfully!")

    def _augment_prompt(self, query, retrieved_docs):
        instruction = "Complete the Python code for the following function."
        
        context = "\n\n### EXAMPLES ###\n"
        for i, doc in enumerate(retrieved_docs):
            context += f"\n# Example {i+1}:\n{doc}\n"
        
        task = f"\n### TASK TO COMPLETE ###\n# Complete the following code:\n{query}"
        
        augmented_prompt = instruction + context + task
        return augmented_prompt

    def generate(self, query, n_results=3, max_new_tokens=100):
        print("Step 1: Retrieving similar prompts from the database...")
        query_embedding = self.embedding_model.encode([query])
        
        retrieved_results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results
        )
        retrieved_prompts = retrieved_results['documents'][0]
        
        print("Step 2: Augmenting the prompt with retrieved context...")
        augmented_prompt = self._augment_prompt(query, retrieved_prompts)
        
        print("Step 3: Generating code...")
        inputs = self.codegen_tokenizer(augmented_prompt, return_tensors="pt")
        outputs = self.codegen_model.generate(**inputs, max_new_tokens=max_new_tokens)
        
        generated_code = self.codegen_tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated_code

In [51]:
pipeline = CodeGenerationPipeline(
    embedding_model=embeddingModel,   
    codegen_model=codeGenModel,          
    codegen_tokenizer=tokenizer,         
    chroma_collection=collections      
)


new_prompt = """
from typing import List

def has_duplicates(numbers: List[float]) -> bool:
    \"\"\"
    Given a list of numbers, return whether the list contains any duplicate numbers.
    \"\"\"
"""

final_code = pipeline.generate(new_prompt)

print("\n\n--- FINAL GENERATED CODE ---")
print(final_code)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Initializing pipeline with existing components...
Pipeline initialized successfully!
Step 1: Retrieving similar prompts from the database...
Step 2: Augmenting the prompt with retrieved context...
Step 3: Generating code...


--- FINAL GENERATED CODE ---
Complete the Python code for the following function.

### EXAMPLES ###

# Example 1:
from typing import List


def remove_duplicates(numbers: List[int]) -> List[int]:
    """ From a list of integers, remove all elements that occur more than once.
    Keep order of elements left the same as in the input.
    >>> remove_duplicates([1, 2, 3, 2, 4])
    [1, 3, 4]
    """


# Example 2:
from typing import List


def rescale_to_unit(numbers: List[float]) -> List[float]:
    """ Given list of numbers (of at least two elements), apply a linear transform to that list,
    such that the smallest number will become 0 and the largest will become 1
    >>> rescale_to_unit([1.0, 2.0, 3.0, 4.0, 5.0])
    [0.0, 0.25, 0.5, 0.75, 1.0]
    """


# Exampl